# Figures and tables of the paper

Produces every data-based figure and table of *Adversarial Upperbounds to Robustness Distributions* in the order in
which they appear in the paper. Figures are written to `plots/`, LaTeX tables to `tables/`. Shared code: `common.py`.

* **MNIST:** α,β-CROWN on the first 100 test images, attacks on the complete test set; "100 images" = image ids 0-99.
* **CIFAR-10:** α,β-CROWN from the JAIR run (3 600 s timeout per query) on a class-balanced sample of 100 test images;
  attacks on the complete test set. "100 images" = this sample (`results/cifar10/balanced_sample_test_indices.csv`).
* **ImageNet:** top-5 attacks on the validation set (used as test set); AutoAttack only on the first 5 000 images,
  "5000 images" = image ids 0-4999.
* **Failed attacks** (no adversarial example in the search space, p* stored as the cap) are excluded from averages,
  standard deviations and statistical tests and reported as failure rates. `EXCLUDE_FAILURES = False` in `common.py`
  includes the cap values, as the submitted manuscript did (this does not reproduce all of its tables, since other
  corrections apply as well).

Not produced here: Fig. 1 (illustration), Table 1 (architectures), Table A1 (name mapping).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st
from IPython.display import display

from common import *

set_style()
pd.set_option("display.max_rows", 200)

## Data

In [ ]:
# MNIST
mnist_df = load_mnist()
mnist_100 = mnist_df[mnist_df.image_id < 100]
mnist_all = mnist_df[mnist_df.verifier != "ABcrown"]

# CIFAR-10
cifar_attacks = load_cifar()
cifar_abcrown = load_cifar_abcrown()
cifar_sample_ids = load_cifar_sample_ids()
cifar_all = snap_cifar_values(cifar_attacks)
cifar_100 = snap_cifar_values(pd.concat([cifar_attacks[cifar_attacks.image_id.isin(cifar_sample_ids)],
                                         cifar_abcrown.drop(columns="original_label")]))

# ImageNet
imagenet_df = load_imagenet()
imagenet_5000 = imagenet_df[imagenet_df.image_id < 5000]
imagenet_all = imagenet_df[imagenet_df.verifier != "AutoAttack"]

for name, df in [("MNIST, 100 images", mnist_100), ("CIFAR-10, 100-image sample", cifar_100),
                 ("CIFAR-10, test set", cifar_all), ("ImageNet, all", imagenet_df)]:
    print(name)
    display(df.groupby(["network", "verifier"]).image_id.nunique().unstack())

In [ ]:
# CIFAR-10: verifier and attacks must refer to the same images. The JAIR file stored the position in the sample as
# image id; it was converted to the test index. Check via the true labels.
raw_attacks = pd.read_csv(DATA_DIR / "cifar10" / "attacks.csv", usecols=["network", "image_id", "original_label"]).drop_duplicates()
raw_attacks["network"] = raw_attacks.network.str.replace("_standard", "", regex=False)
label_check = cifar_abcrown.merge(raw_attacks, on=["network", "image_id"], suffixes=("_abcrown", "_attacks"))
print(f"labels equal: {(label_check.original_label_abcrown == label_check.original_label_attacks).sum()} / {len(label_check)}")
assert (label_check.original_label_abcrown == label_check.original_label_attacks).all()

## Table 2: monotonicity

Iterative search over every ε of the search space. A violation means that the attack failed at a larger ε than the
smallest ε at which it succeeded; the step distance is the number of steps between the highest failed and the lowest
successful ε.

* MNIST: correctly classified images among the first 100 test images, four attacks (FAB was not part of this experiment).
* CIFAR-10: 20 images per network from a class-balanced sample of 100 **training** images.
* ImageNet: 20 consecutive validation images per network; the PGD and AutoAttack runs hit the job time limit, so
  fewer images were evaluated. openai_2 was not evaluated.

In [ ]:
def monotonicity_results(df, step):
    df = df.copy()
    df["step_distance"] = ((df.epsilon_value - df.smallest_sat_value) / step).round()
    df["violation"] = df.step_distance > 0
    return df

def monotonicity_table(df):
    violations = df[df.violation]
    table = df.groupby(["verifier", "network"]).agg(evaluated_images=("image_id", "size"), number_images=("violation", "sum"))
    distances = violations.groupby(["verifier", "network"]).step_distance.agg(["mean", "median", "max"])
    return table.join(distances).fillna({"mean": 0, "median": 0, "max": 0})

mnist_mono_df = monotonicity_results(load_mnist_monotonicity(), MNIST_STEP)
cifar_mono_df = monotonicity_results(load_cifar_monotonicity(), CIFAR_STEP)
imagenet_mono_df = monotonicity_results(load_imagenet_monotonicity(), IMAGENET_STEP)
for name, df in [("MNIST", mnist_mono_df), ("CIFAR-10", cifar_mono_df), ("ImageNet", imagenet_mono_df)]:
    print(f"{name}: {len(df)} runs, {df.violation.sum()} violations, max distance {df.step_distance.max():.0f}")
    display(df[df.violation][["verifier", "network", "image_id", "step_distance"]])

In [ ]:
mono_table = pd.concat({"MNIST": monotonicity_table(mnist_mono_df), "CIFAR-10": monotonicity_table(cifar_mono_df),
                        "ImageNet": monotonicity_table(imagenet_mono_df)})
display(mono_table)

# LaTeX in the layout of the paper: one block per dataset, attack name on the first row, network names as in the paper,
# networks in the same order as in the figures (Figures 2, 3 and 4)
paper_rows = {
    "MNIST": [(a, MNIST_ORDER) for a in ["AutoAttack", "PGD", "PGD random", "FGSM"]],
    "CIFAR-10": [(a, CIFAR_ORDER) for a in ["AutoAttack", "PGD", "FGSM"]],
    "ImageNet": [(a, IMAGENET_ORDER) for a in ["AutoAttack", "PGD", "FGSM"]],
}
paper_network_names = {"mnist-net_256x2": r"mnist$-$net\_256x2", "conv_big": r"conv\_big\_standard",
                       "resnet_4b": r"resnet\_4b\_standard"}
paper_attack_names = {"PGD random": r"PGD\_random"}
columns = ["number images", "step distance mean"]   # evaluated images and median/max distance: see display above

def number(x):
    return f"{int(round(x)):,}".replace(",", r"\,")

lines = [r"\begin{tabularx}{\textwidth}{Xlrr}", r"\toprule",
         "attack & network & " + " & ".join(columns) + r" \\", r"\midrule"]
for i, (dataset, blocks) in enumerate(paper_rows.items()):
    if i > 0:
        lines.append(r"\midrule")
    lines += [rf"\multicolumn{{{len(columns) + 2}}}{{c}}{{{dataset}}} \\", r"\midrule"]
    for attack, networks in blocks:
        for j, network in enumerate(networks):
            label = paper_attack_names.get(attack, attack) if j == 0 else ""
            name = r"\texttt{" + paper_network_names.get(network, network.replace("_", r"\_")) + "}"
            if (attack, network) in mono_table.loc[dataset].index:
                row = mono_table.loc[(dataset, attack, network)]
                values = " & ".join(number(row[c]) for c in ["number_images", "mean"])
            else:   # openai_2 was not evaluated in the monotonicity experiment
                values = rf"\multicolumn{{{len(columns)}}}{{c}}{{not evaluated}}"
            lines.append(f"{label} & {name} & {values} " + r"\\")
lines += [r"\bottomrule", r"\end{tabularx}"]
print(write_paper_table("table_2", "tab:monotonicity", lines))

## Figure 2: MNIST, first 100 images

In [ ]:
fig, axes = plt.subplots(2, 2)
for ax, network in zip(np.ravel(axes), MNIST_ORDER):
    t_df = mnist_100[mnist_100.network == network]
    plot_cdfs(t_df, "algorithm", ax, network, FULL_XLIM, hue_order=ALGORITHM_ORDER, xtick=0.1, labels=PAPER_LEGEND_LABELS)
    plot_verification_gap(t_df[t_df.verifier == "ABcrown"], ax, color=sns.color_palette("Paired")[0])
remove_legends_except(axes, axes[0][1])
sns.move_legend(axes[0][1], "upper right")
plt.tight_layout()
plt.savefig(PLOT_DIR / "anneplot_comparison.eps")

## Figure 3: CIFAR-10, 100-image sample

In [ ]:
fig, axes = plt.subplots(2, 2)
for ax, network in zip(np.ravel(axes), CIFAR_ORDER):
    t_df = cifar_100[cifar_100.network == network]
    plot_cdfs(t_df, "algorithm", ax, network, (0, 0.2), hue_order=CIFAR_ALGORITHMS, xtick=0.05)
    plot_verification_gap(t_df[t_df.verifier == "ABcrown"], ax, color=sns.color_palette("Paired")[0])
axes[1][1].axis("off")
remove_legends_except(axes, axes[1][0])
sns.move_legend(axes[1][0], "lower right")
plt.tight_layout()
plt.savefig(PLOT_DIR / "anneplot_comparison_cifar.eps")

## Figure 4: ImageNet, attacks per network (5000 images)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(11.7, 12))
positions = [(0, 0), (0, 1), (1, 0), (1, 1), (2, 0)]
for (row, col), network in zip(positions, IMAGENET_ORDER):
    plot_cdfs(imagenet_5000[imagenet_5000.network == network], "algorithm", axes[row][col], network, FULL_XLIM,
              hue_order=["AutoAttack", "PGD", "FGSM"], xtick=0.1)
axes[2][1].axis("off")
remove_legends_except(axes, axes[2][0])
sns.move_legend(axes[2][0], "center left")
plt.tight_layout()
plt.savefig(PLOT_DIR / "imagenet_attack_comparison.eps")

## Figure 5: α,β-CROWN vs. AutoAttack (top pane MNIST, bottom pane CIFAR-10; two files as in the paper)

In [ ]:
panes = [("ab_crown_autoattack_correlation.eps", mnist_100, ["mnist_relu_4_1024", "mnist-net_256x2"], (0, 0.2), 0.05),
         ("cifar_autoattack_vs_abcrown.eps", cifar_100, ["cifar_7_1024", "conv_big"], (0, 0.03), 0.005)]
with plt.rc_context(SANS):
    for file_name, df, networks, lim, xtick in panes:
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        for ax, network in zip(axes, networks):
            scatter_with_spearman(df[df.network == network], "ABcrown", "AutoAttack", ax, lim, xtick=xtick)
            ax.set_title(network)
        plt.tight_layout()
        plt.savefig(PLOT_DIR / file_name)

## Table 3: performance comparison (MNIST, CIFAR-10, ImageNet)

In [ ]:
blocks = []
for title, subset_label, subset, full in [("MNIST networks", "100 images", mnist_100, mnist_all),
                                          ("CIFAR-10 networks", "100 images", cifar_100, cifar_all),
                                          ("ImageNet networks", "5000 images", imagenet_5000, imagenet_all)]:
    blocks.append((title, subset_label, performance_table(subset), performance_table(full)))
    display(pd.concat({subset_label: blocks[-1][2], "Complete test set": blocks[-1][3]}, axis=1))
print(write_paper_table("table_3", "tab:performance", tex_performance_table(blocks)))

## Table 4: instances with an optimal upper bound (MNIST, CIFAR-10)

In [ ]:
blocks = []
for name, df, step in [("MNIST", mnist_100, MNIST_STEP), ("CIFAR-10", cifar_100, CIFAR_STEP)]:
    table_4, n_tight = tight_instances_table(df, step)
    display(table_4)
    blocks.append((f"{name} ({n_tight} instances across the {df.network.nunique()} networks)", table_4))
print(write_paper_table("table_4", "tab:statistics:tight", tex_tight_table(blocks)))

Section 5.4: cases where α,β-CROWN verified robustness at a radius at which an attack found an adversarial example
(attack p* ≤ α,β-CROWN ε*). Must be empty on matched images (reviewer comments B.1/B.2).

In [ ]:
for name, df in [("MNIST", mnist_100), ("CIFAR-10", cifar_100)]:
    eps_star = df[df.verifier == "ABcrown"][["network", "image_id", "epsilon_value"]]
    contradictions = df[df.verifier != "ABcrown"].merge(eps_star, on=["network", "image_id"], suffixes=("", "_abcrown"))
    contradictions = contradictions[contradictions.smallest_sat_value <= contradictions.epsilon_value_abcrown]
    print(f"{name}: {len(contradictions)} contradicting results")

## Figure 6: MNIST, 100 images vs. complete test set (PGD)

In [ ]:
merged = pd.concat([mnist_100.assign(dataset="100 images"), mnist_all.assign(dataset="test set")])
fig, axes = plt.subplots(2, 2)
for ax, network in zip(np.ravel(axes), MNIST_ORDER):
    plot_cdfs(merged[(merged.network == network) & (merged.verifier == "PGD")], "dataset", ax, network, FULL_XLIM, xtick=0.1)
remove_legends_except(axes, axes[1][1])
sns.move_legend(axes[1][1], "lower right")
plt.tight_layout()
plt.savefig(PLOT_DIR / "PGD_full_dataset_visual_comparison.eps")

## Figure 7: CIFAR-10, 100-image sample vs. complete test set (PGD)

In [ ]:
merged = pd.concat([cifar_100.assign(dataset="100 images"), cifar_all.assign(dataset="test set")])
fig, axes = plt.subplots(2, 2)
for ax, network in zip(np.ravel(axes), CIFAR_ORDER):
    plot_cdfs(merged[(merged.network == network) & (merged.verifier == "PGD")], "dataset", ax, network, (0, 0.2), xtick=0.05)
    sns.move_legend(ax, "upper right")
axes[1][1].axis("off")
plt.tight_layout()
plt.savefig(PLOT_DIR / "cifar_10_to_full.eps")

## Figure 8: ImageNet, 5000 images vs. complete test set (PGD)

In [ ]:
merged = pd.concat([imagenet_5000.assign(dataset="5000 images"), imagenet_all.assign(dataset="test set")])
fig, axes = plt.subplots(2, 2)
for ax, network in zip(np.ravel(axes), ["eva", "beitv2", "openai_2", "laion_2b"]):
    plot_cdfs(merged[(merged.network == network) & (merged.verifier == "PGD")], "dataset", ax, network, (0, 0.2), xtick=0.05)
remove_legends_except(axes, axes[1][1])
sns.move_legend(axes[1][1], "upper right")
plt.tight_layout()
plt.savefig(PLOT_DIR / "imagenet_5000_vs_full.eps")

## Figure 9: ImageNet, top-1 vs. top-5 (openai_1)

In [ ]:
top1_df = load_imagenet_top1()
t_df = pd.concat([top1_df, imagenet_df[imagenet_df.network == "openai_1"]])
t_df = t_df[(t_df.network == "openai_1") & (t_df.image_id < 5000)]
t_df["label"] = t_df.verifier + ", " + t_df.top_k
# Included at 0.6\linewidth (Fig. 2 etc.: 2x2 at 11.7 in and \linewidth), so 7.0 in gives the same font size
fig, ax = plt.subplots(figsize=(7.0, 4.67))
plot_cdfs(t_df, "label", ax, "openai_1", FULL_XLIM, xtick=0.05,
          hue_order=["PGD, top-1", "AutoAttack, top-1", "FGSM, top-1", "AutoAttack, top-5", "PGD, top-5", "FGSM, top-5"])
sns.move_legend(ax, "lower right")
plt.tight_layout()
plt.savefig(PLOT_DIR / "top_1_vs_top_5_imagenet.eps")
t_df.groupby("label").image_id.nunique()

## Figure 10: comparison of the networks (PGD, complete test sets)

In [ ]:
fig, axes = plt.subplots(2, 2)
panels = [(axes[0][0], mnist_all, MNIST_ORDER, FULL_XLIM, 0.1, "lower right", "(a) MNIST"),
          (axes[0][1], cifar_all, CIFAR_ORDER, (0, 0.2), 0.05, "upper right", "(b) CIFAR-10"),
          (axes[1][0], imagenet_df, IMAGENET_ORDER, (0, 0.1), 0.02, "upper right", "(c) ImageNet")]
for ax, df, order, xlim, xtick, legend_loc, title in panels:
    sns.ecdfplot(df[df.verifier == "PGD"], x="smallest_sat_value", hue="network", hue_order=order, ax=ax)
    ax.set_xlim(*xlim)
    ax.xaxis.set_major_locator(plt.MultipleLocator(xtick))
    sns.move_legend(ax, legend_loc)
    ax.set_title(title)
    ax.set_xlabel("Epsilon value")
    ax.set_ylabel("Fraction $p*$ values found")
axes[1][1].axis("off")
plt.tight_layout()
plt.savefig(PLOT_DIR / "network_comparison_all_datasets.eps")

## Appendix C: running time and p* statistics (Tables C2-C7)

C6 (complete ImageNet test set) contains PGD and FGSM only: AutoAttack was evaluated on the first 5 000 images only
(C7). The same holds for D16 and the complete-test-set columns of Table 3 (ImageNet).

In [ ]:
# Decimals as in the paper: (time, p*); MNIST times with thousands separators
for name, label, df, decimals in [("C2", "tab:total_times_complete_mnist", mnist_all, (2, 3, True)),
                                  ("C3", "tab:total_times_combined", mnist_100, (2, 3, True)),
                                  ("C4", "tab:total_times_complete_cifar", cifar_all, (3, 3, False)),
                                  ("C5", "tab:total_times_combined_cifar", cifar_100, (3, 3, False)),
                                  ("C6", "tab:total_times_complete_imagenet", imagenet_all, (4, 4, False)),
                                  ("C7", "tab:total_times_combined_imagenet", imagenet_5000, (4, 4, False))]:
    table = statistics_table(df)
    print(f"Table {name}")
    display(table)
    write_paper_table(f"table_{name}", label, tex_statistics_table(table, *decimals))

## Table D8: K-S test for log-normality of the α,β-CROWN distributions (MNIST)

In [ ]:
abcrown_df = mnist_100[mnist_100.verifier == "ABcrown"]
upper = lognormal_ks_table(abcrown_df, "smallest_sat_value")
lower = lognormal_ks_table(abcrown_df, "epsilon_value")
display(upper, lower)
print(write_paper_table("table_D8", "fig:abcrown_ks_tests", tex_abcrown_lognormal_table(upper, lower)))

## Figure D1: pairwise t-tests on log(p*), MNIST, first 100 images

In [ ]:
def pairwise_ttest(df, verifier_names):
    statistic_df = pd.DataFrame(index=verifier_names, columns=verifier_names, dtype=float)
    mask_df = pd.DataFrame(index=verifier_names, columns=verifier_names, dtype=float)
    for a in verifier_names:
        for b in verifier_names:
            statistic, p_value = st.ttest_ind(np.log(df[df.verifier == a].smallest_sat_value),
                                              np.log(df[df.verifier == b].smallest_sat_value),
                                              alternative="two-sided", equal_var=False)
            statistic_df.loc[a, b] = statistic
            mask_df.loc[a, b] = 1 if p_value >= 0.05 else 0
    return statistic_df, mask_df

cmap = LinearSegmentedColormap.from_list("", [(0, sns.color_palette("Paired", 8)[0]), (.05, sns.color_palette("Paired", 8)[6]), (1, sns.color_palette("Paired", 8)[6])])
with plt.rc_context(SANS):
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    for ax, network in zip(np.ravel(axes), MNIST_ORDER):
        statistic_df, mask_df = pairwise_ttest(successful(mnist_100[mnist_100.network == network]), ALGORITHM_ORDER)
        sns.heatmap(to_raw_names(mask_df), annot=to_raw_names(statistic_df), fmt=".2f", cmap=cmap, vmin=0.0, vmax=1.0, cbar=False, ax=ax)
        ax.set_title(network)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "pairwise_t_tests_100.eps")

## Tables D9, D10: K-S tests for log-normality (MNIST)

In [ ]:
for name, label, df in [("D9", "fig:attack_100_log_normal_ks_tests", mnist_100),
                        ("D10", "fig:attack_all_images_log_normal_tests_mnist", mnist_all)]:
    table = lognormal_ks_table(df)
    display(table)
    write_paper_table(f"table_{name}", label, tex_grouped_test_table(
        table, "network", ["statistic", "p-value", "passed"], ["network", "algorithm", "statistic", "p-value", "passed"]))

## Table D11: MNIST, first 100 images vs. rest of the test set

The 100 images are removed from the test set so both samples are disjoint (two-sample K-S assumption, B.13).
`p-value (overlapping)` is the comparison as in the submitted paper (subset contained in the test set).

In [ ]:
def subset_vs_rest(subset, full, algorithms=None, network_order=None):
    table = subset_vs_full_ks_table(subset, full, algorithms=algorithms, network_order=network_order)
    overlapping = subset_vs_full_ks_table(subset, full, exclude_subset=False, algorithms=algorithms, network_order=network_order)
    table = table.merge(overlapping[["algorithm", "network", "p-value"]].rename(columns={"p-value": "p-value (overlapping)"}),
                        on=["algorithm", "network"])
    display(table)
    return table[["algorithm", "network", "p-value", "passed"]]

mnist_attacks = [a for a in ALGORITHM_ORDER if a != "ABcrown"]
table_d11 = subset_vs_rest(mnist_100, mnist_all, algorithms=mnist_attacks)
print(write_paper_table("table_D11", "fig:attack_100_vs_all_ks_tests", tex_grouped_test_table(table_d11, "algorithm", ["p-value", "passed"], ["algorithm", "network", "p-value", "passed"])))

## Figure D2: pairwise K-S tests, MNIST, complete test set

In [ ]:
with plt.rc_context(SANS):
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    for ax, network in zip(np.ravel(axes), MNIST_ORDER):
        plot_pairwise_ks(mnist_all[mnist_all.network == network], mnist_attacks, ax, network)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "pairwise_ks_tests_all.eps")

## Tables D12, D13: K-S tests for log-normality (CIFAR-10)

In [ ]:
for name, label, df in [("D12", "fig:attack_100_log_normal_ks_tests_cifar", cifar_100),
                        ("D13", "fig:attack_all_images_log_normal_tests_imagenet_cifar", cifar_all)]:
    table = lognormal_ks_table(df)
    display(table)
    write_paper_table(f"table_{name}", label, tex_grouped_test_table(
        table, "network", ["statistic", "p-value", "passed"], ["network", "algorithm", "statistic", "p-value", "passed"]))

## Table D14: CIFAR-10, 100-image sample vs. rest of the test set

In [ ]:
table_d14 = subset_vs_rest(cifar_100, cifar_all, network_order=["cifar_7_1024", "resnet_4b", "conv_big"])
print(write_paper_table("table_D14", "fig:attack_100_vs_all_ks_tests_cifar", tex_grouped_test_table(table_d14, "algorithm", ["p-value", "passed"], ["algorithm", "network", "p-value", "passed"])))

## Tables D15, D16: K-S tests for log-normality (ImageNet)

In the submitted paper D16 duplicated D15 (reviewer comment B.5). D16 contains PGD and FGSM only, since AutoAttack
was run on the first 5 000 images only.

In [ ]:
for name, label, df in [("D15", "fig:attack_100_log_normal_ks_tests_imagenet", imagenet_5000),
                        ("D16", "fig:attack_all_images_log_normal_tests_imagenet", imagenet_all)]:
    table = lognormal_ks_table(df)
    display(table)
    write_paper_table(f"table_{name}", label, tex_grouped_test_table(
        table, "network", ["statistic", "p-value", "passed"], ["network", "algorithm", "stat.", "p-value", "passed"]))

## Table D17: ImageNet, first 5000 images vs. rest of the test set

In [ ]:
table_d17 = subset_vs_rest(imagenet_5000, imagenet_all)
print(write_paper_table("table_D17", "fig:attack_100_vs_all_ks_tests_imagenet", tex_grouped_test_table(table_d17, "algorithm", ["p-value", "passed"], ["algorithm", "network", "p-value", "passed"])))

## New appendix table: sample sizes (reviewer B.17)

For every dataset, experiment and network: initial images, images removed because they are misclassified (top-1 for
MNIST/CIFAR-10, top-5 for ImageNet), retained images, retained images without a result for a method ("not
evaluated", in the overview only), and the images with a result for every method ("used"). "Used" is the denominator of all averages,
failure rates and RMC values in Tables 3, 4 and C2–C7; the figures show each method on all its evaluated images;
Table 2 counts violations among the evaluated images of each method. Written as two complete `table` environments
for the appendix: `tables/table_experiments.tex` (the experiments: initial images, methods, radii, where they are
used) and `tables/table_sample_sizes.tex` (images per network).

In [ ]:
def size_rows(df, initial, order, used=True):
    counts = df.groupby(["network", "verifier"]).image_id.nunique().unstack()
    used_counts = restrict_to_common_instances(df).groupby("network").image_id.nunique()
    rows = []
    for network in order:
        if network not in counts.index:   # openai_2 in the ImageNet monotonicity experiment
            rows.append([network, initial, None, None, None, None])
            continue
        evaluated = counts.loc[network].dropna()
        retained = int(evaluated.max())
        missing = [(method, retained - int(n)) for method, n in evaluated.items() if n < retained]
        rows.append([network, initial, None if initial is None else initial - retained, retained,
                     missing, int(used_counts[network]) if used else None])
    return rows

top1_openai = load_imagenet_top1()
top1_openai = top1_openai[(top1_openai.network == "openai_1") & (top1_openai.image_id < 5000)]
mono = {"MNIST": mnist_mono_df, "CIFAR-10": cifar_mono_df, "ImageNet": imagenet_mono_df}
# (dataset, experiment, methods, radii, used in, rows)
experiments = [
    ("MNIST", "First 100 test images", 7, 102, "Figs. 2, 5, 6, D1; Tables 3, 4, C3, D8, D9, D11", size_rows(mnist_100, 100, MNIST_ORDER)),
    ("MNIST", "Complete test set", 6, 102, "Figs. 6, 10, D2; Tables 3, C2, D10, D11", size_rows(mnist_all, 10000, MNIST_ORDER)),
    ("MNIST", "Monotonicity (iterative search)", 4, 102, "Table 2", size_rows(mono["MNIST"], 100, MNIST_ORDER, used=False)),
    ("CIFAR-10", "Balanced sample of 100 test images", 4, 103, "Figs. 3, 5, 7; Tables 3, 4, C5, D12, D14", size_rows(cifar_100, 100, CIFAR_ORDER)),
    ("CIFAR-10", "Complete test set", 3, 103, "Figs. 7, 10; Tables 3, C4, D13, D14", size_rows(cifar_all, 10000, CIFAR_ORDER)),
    ("CIFAR-10", "Monotonicity (20 training images per network)", 3, 103, "Table 2", size_rows(mono["CIFAR-10"], None, CIFAR_ORDER, used=False)),
    ("ImageNet", "First 5\\,000 validation images (top-5)", 3, 4000, "Figs. 4, 8, 9; Tables 3, C7, D15, D17", size_rows(imagenet_5000, 5000, IMAGENET_ORDER)),
    ("ImageNet", "Validation set (top-5)", 2, 4000, "Figs. 8, 10; Tables 3, C6, D16, D17", size_rows(imagenet_all, 50000, IMAGENET_ORDER)),
    ("ImageNet", "First 5\\,000 validation images (top-1)", 3, 4000, "Fig. 9", size_rows(top1_openai, 5000, ["openai_1"])),
    ("ImageNet", "Monotonicity (20 validation images per network)", 3, 4000, "Table 2", size_rows(mono["ImageNet"], None, IMAGENET_ORDER, used=False)),
]

columns = ["network", "initial", "misclassified", "retained", "not evaluated", "used"]
overview = pd.concat({(d, e): pd.DataFrame(rows, columns=columns).set_index("network") for d, e, _, _, _, rows in experiments})
count_columns = ["initial", "misclassified", "retained", "used"]
overview[count_columns] = overview[count_columns].astype("Int64")   # integer counts; "<NA>" where not applicable
overview["not evaluated"] = overview["not evaluated"].apply(lambda m: "not evaluated" if m is None else ", ".join(f"{a} {n:,}" for a, n in m))
display(overview)

def count(n):
    return "-" if n is None else f"{n:,}".replace(",", r"\,")


# Paper layout: two tables. tab:experiments describes the experiments (one row each), tab:sample_sizes gives the
# number of images per network for the three experiments of each dataset side by side.
role = {"First 100 test images": "Subset", "Balanced sample of 100 test images": "Subset",
        "First 5\\,000 validation images (top-5)": "Subset", "Complete test set": "Test set",
        "Validation set (top-5)": "Test set", "First 5\\,000 validation images (top-1)": "Top-1"}
descriptions = {
    "First 100 test images": "first 100 test images",
    "Complete test set": "complete test set",
    "Monotonicity (iterative search)": "first 100 test images, iterative search",
    "Balanced sample of 100 test images": "class-balanced sample of 100 test images",
    "Monotonicity (20 training images per network)": "20 training images per network (from a class-balanced sample of 100)",
    "First 5\\,000 validation images (top-5)": "first 5\\,000 validation images",
    "Validation set (top-5)": "validation set (used as test set)",
    "First 5\\,000 validation images (top-1)": "first 5\\,000 validation images, top-1, \\texttt{openai\\_1} only",
    "Monotonicity (20 validation images per network)": "20 validation images per network",
}
initial_images = {"Monotonicity (20 training images per network)": "20", "Monotonicity (20 validation images per network)": "20"}

experiment_lines = [r"\begin{table}[tbp]", r"  \centering",
    r"  \caption{Experiments per dataset. \emph{Initial}: images considered per network; \emph{methods}: verifier "
    r"and attacks compared; \emph{radii}: number of $\varepsilon$-values in the search space. Misclassified images "
    r"(top-1 for MNIST and CIFAR-10, top-5 for ImageNet) are removed, see Table~\ref{tab:sample_sizes}. For the "
    r"monotonicity experiments on CIFAR-10 and ImageNet, 20 consecutive correctly classified images were selected "
    r"per network.}",
    r"  \label{tab:experiments}", r"  \small",
    # tabularx (loaded by the paper, also loads array): ragged-right text columns, full line width
    r"  \begin{tabularx}{\linewidth}{>{\raggedright\arraybackslash}Xrrr>{\raggedright\arraybackslash}p{3.6cm}}",
    r"  \toprule",
    r"  experiment & initial & methods & radii & used in \\"]
dataset_prev = None
for dataset, experiment, methods, radii, used_in, rows in experiments:
    if dataset != dataset_prev:
        experiment_lines += [r"  \midrule", rf"  \multicolumn{{5}}{{l}}{{\textbf{{{dataset}}}}} \\"]
        dataset_prev = dataset
    initial = initial_images.get(experiment, count(rows[0][1]))
    label = role.get(experiment, "Monotonicity")
    experiment_lines.append(rf"  \emph{{{label}}}: {descriptions[experiment]} & {initial} & {methods} & {count(radii)} & {used_in} \\")
experiment_lines += [r"  \bottomrule", r"  \end{tabularx}", r"\end{table}"]

blocks = {}
for dataset, experiment, methods, radii, used_in, rows in experiments:
    blocks.setdefault(dataset, []).append({"experiment": experiment, "rows": {r[0]: r for r in rows}})
orders = {"MNIST": MNIST_ORDER, "CIFAR-10": CIFAR_ORDER, "ImageNet": IMAGENET_ORDER}
top1 = blocks["ImageNet"][2]["rows"]["openai_1"]   # Fig. 9: top-1 experiment on openai_1

size_lines = [r"\begin{table}[tbp]", r"  \centering",
    r"  \caption{Number of images per network for the experiments of Table~\ref{tab:experiments}. \emph{Retained}: "
    r"correctly classified images (initial minus misclassified); \emph{used}: images with a result for every method, "
    r"the denominator of all averages, failure rates and RMC values in Tables~3, 4 and C2--C7. For some methods, "
    r"results are missing for individual retained images (e.g.\ \abcrown{} on CIFAR-10, AutoAttack on \texttt{eva} "
    r"and \texttt{laion\_2b}); figures show each method on all its evaluated images. In the ImageNet monotonicity "
    r"experiment, PGD and AutoAttack were evaluated on 5--15 and 1--4 of the 20 images. "
    rf"Top-1 experiment (Fig.~9, \texttt{{openai\_1}}): {count(top1[3])} retained, AutoAttack evaluated on "
    rf"{count(top1[5])}.}}",
    r"  \label{tab:sample_sizes}", r"  \small",
    r"  \begin{tabular}{lrrrrr}", r"  \toprule",
    r"  & \multicolumn{2}{c}{subset} & \multicolumn{2}{c}{test set} & monotonicity \\",
    r"  \cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(lr){6-6}",
    r"  network & retained & used & retained & used & images \\"]
for dataset, (subset, test_set, *_, monotonicity) in blocks.items():
    size_lines += [r"  \midrule", rf"  \multicolumn{{6}}{{l}}{{\textbf{{{dataset}}}}} \\"]
    for network in orders[dataset]:
        s_row, t_row, m_row = subset["rows"][network], test_set["rows"][network], monotonicity["rows"][network]
        mono = "-" if m_row[3] is None else count(m_row[3])
        size_lines.append(rf"  {tex_network(network)} & {count(s_row[3])} & {count(s_row[5])} & {count(t_row[3])} & {count(t_row[5])} & {mono} \\")
size_lines += [r"  \bottomrule", r"  \end{tabular}", r"\end{table}"]

(TABLE_DIR / "table_experiments.tex").write_text("\n".join(experiment_lines) + "\n")
(TABLE_DIR / "table_sample_sizes.tex").write_text("\n".join(size_lines) + "\n")
print("\n".join(experiment_lines)); print(); print("\n".join(size_lines))